# 1. Data preparation

**Question of the series:** on which questions does graph-based retrieval beat plain vector retrieval, and at what cost?

Four systems answer the same questions from the same documents:

| System | Retrieval idea |
|---|---|
| `naive_rag` | Embed text chunks, return the closest ones. |
| `lightrag_hybrid` | Build an entity-relation graph, retrieve entities and relations matching the question's keywords. |
| `graphrag_local` | Build a graph with communities, start from the entities closest to the question. |
| `graphrag_global` | Summarise every community, then combine the summaries (map-reduce). |

This notebook prepares the questions and documents that all four systems share.

In [1]:
from dotenv import load_dotenv

from src import config

load_dotenv(config.PROJECT_ROOT / ".env")
print(f"Domain: {config.DOMAIN} | run mode: {config.RUN_MODE.value}")

Domain: technology | run mode: subset


## 1.1 Download the benchmark

[WildGraphBench](https://arxiv.org/abs/2602.02053) is built from Wikipedia. Each question comes from a Wikipedia statement, and the corpus is the set of web pages that statement cites. These pages are long, noisy and heterogeneous: navigation menus, cookie banners and archived layouts are left as is. That is the "wild" part of the benchmark.

The download is limited to one domain and pinned to one dataset revision.

In [2]:
from src.data import download_wildgraphbench_domain

download_wildgraphbench_domain(
    domain=config.DOMAIN,
    raw_data_directory=config.RAW_DATA_DIRECTORY,
    repository_id=config.DATASET_REPOSITORY_ID,
    revision=config.DATASET_REVISION,
)
sorted(path.name for path in (config.RAW_DATA_DIRECTORY / "corpus" / config.DOMAIN).iterdir())

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 444 files:   0%|          | 0/444 [00:00<?, ?it/s]

['Steam(service)']

## 1.2 Three kinds of questions

- **Single-fact:** the answer sits in one cited page.
- **Multi-fact:** the answer combines facts from at least two pages.
- **Summary:** the answer should cover a whole Wikipedia section, graded fact by fact against `gold_statements`.

The paper reports its results per type, and so does this benchmark.

In [3]:
from src.data import load_domain_questions

all_questions = load_domain_questions(config.RAW_DATA_DIRECTORY, config.DOMAIN)
all_questions["question_type"].value_counts()

question_type
single_fact    56
multi_fact     33
summary        24
Name: count, dtype: int64

In [4]:
for question_type, type_questions in all_questions.groupby("question_type", sort=False):
    example = type_questions.iloc[0]
    print(f"--- {question_type} ({len(example['ref_urls'])} cited pages)")
    print("Q:", example["question"])
    if question_type == "summary":
        for statement in example["gold_statements"]:
            print("  -", statement)
    else:
        print("A:", example["answer"])
    print()

--- single_fact (1 cited pages)
Q: What figures for peak simultaneous players, along with daily and monthly user activity, did Valve announce for Steam in August 2017?
A: By August 2017, Valve reported a peak of 14 million concurrent players (up from 8.4 million in 2015), with 33 million daily active users and 67 million monthly active users.

--- multi_fact (2 cited pages)
Q: In 2018, how did Valve first address and then later clarify its stance on games and creators engaging in 'trolling' behavior?
A: Beginning in June 2018, Valve took action against "trolling" games and developers, and in September 2018, defined trolls as those not making good faith efforts to create and sell games.

--- summary (6 cited pages)
Q: How has Valve approached quality control and disallowed functionality, particularly in response to developers submitting manipulative or bad-faith games?
  - Beginning in June 2018, Valve took action against "trolling" games and developers, and in September 2018, defined t

## 1.3 The corpus

The corpus is made of the cited web pages only. The Wikipedia article itself (`<topic>.txt`) is left out, because the gold answers were written from it: indexing it would hand every system the answer key.

In [5]:
import plotly.express as px

from src.data import load_domain_documents

all_documents = load_domain_documents(config.RAW_DATA_DIRECTORY, config.DOMAIN, config.TOKENIZER_ENCODING)
print(f"{len(all_documents)} pages, {all_documents['token_count'].sum():,} tokens")
all_documents[["title", "token_count"]].sort_values("token_count", ascending=False).head()

441 pages, 3,351,390 tokens


,title,token_count
254,Steam'd Penguins,230946
296,Thought: Do We Own Our Steam Games?,91735
180,Steam Users See Big Problems With Charging For...,83352
45,Faster Zombies!,53640
57,Gabe Newell On Removing Valve From Steam,51001


In [6]:
figure = px.histogram(
    all_documents,
    x="token_count",
    nbins=60,
    log_x=False,
    title="Page length in tokens",
    color_discrete_sequence=["#2a78d6"],
    template="plotly_white",
)
figure.update_layout(bargap=0.05, xaxis_title="Tokens per page", yaxis_title="Pages")
figure.show()

Most pages hold a few thousand tokens, but a handful exceed 100,000. At 1,200 tokens per chunk, the full corpus yields about 3,000 chunks, and every chunk costs one or more LLM calls in the graph pipelines.

## 1.4 Link questions to their evidence

Each question lists the URLs it relies on. Mapping those URLs to pages tells whether the evidence of a question is actually in the corpus. A few pages were never saved by the dataset authors.

In [7]:
from src.data import attach_cited_documents, map_reference_urls_to_documents

url_to_document_id = map_reference_urls_to_documents(config.RAW_DATA_DIRECTORY, config.DOMAIN, all_documents)
questions_with_evidence = attach_cited_documents(all_questions, url_to_document_id)
questions_with_evidence.groupby("question_type")["all_citations_available"].agg(["sum", "size"]).rename(
    columns={"sum": "evidence_complete", "size": "questions"}
)

,evidence_complete,questions
question_type,,
multi_fact,32,33
single_fact,56,56
summary,24,24


## 1.5 Select the questions and documents of this run

`config.RUN_MODE` decides the size of the run:

- `SUBSET` draws a few questions per type among those with complete evidence, and keeps only the pages they cite. It validates the pipeline and measures the cost per token for a few cents. Its scores are **not** meaningful: the corpus contains almost nothing but relevant pages.
- `FULL` keeps all questions and all pages. This is the benchmark.

The files written here are read by the next notebooks.

In [8]:
from src.data import load_full_corpus_statistics, prepare_run_inputs

run_inputs = prepare_run_inputs(
    raw_data_directory=config.RAW_DATA_DIRECTORY,
    domain=config.DOMAIN,
    run_mode=config.RUN_MODE,
    subset_questions_per_type=config.SUBSET_QUESTIONS_PER_TYPE,
    random_seed=config.RANDOM_SEED,
    encoding_name=config.TOKENIZER_ENCODING,
)
full_corpus_statistics = load_full_corpus_statistics(config.DOMAIN)
print(f"Run folder: {run_inputs.run_directory}")
print(f"Questions: {len(run_inputs.questions)} of {full_corpus_statistics['question_count']}")
print(f"Documents: {len(run_inputs.documents)} of {full_corpus_statistics['document_count']}")
print(f"Tokens:    {run_inputs.documents['token_count'].sum():,} of {full_corpus_statistics['token_count']:,}")
run_inputs.questions[["question_id", "question_type", "question"]]

Run folder: /Users/linafaik/Documents/projects/graph-retrieval-bench/outputs/technology/subset
Questions: 9 of 113
Documents: 23 of 441
Tokens:    166,551 of 3,351,390


,question_id,question_type,question
0,technology-000,single_fact,"What figures for peak simultaneous players, al..."
1,technology-005,single_fact,What was the economic outcome of a Steam featu...
2,technology-033,single_fact,What judicial decision was reached in May 2022...
3,technology-071,multi_fact,"For indie developers unable to cover the cost,..."
4,technology-081,multi_fact,What was Valve's original intention for implem...
5,technology-086,multi_fact,What were the two consecutive player count rec...
6,technology-089,summary,How has Valve approached quality control and d...
7,technology-097,summary,"What is the Steam service, and how has it grow..."
8,technology-105,summary,Describe the history and strategic significanc...


## Next

The same questions and documents now feed three indexes. Notebook 02 starts with the simplest one: vector search over chunks.